In [ ]:
# user deploy (2)
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex( # search engine
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [3]:
query_vector = model.encode("How do I run Kafka?") # we need to encode in (2)
results = vs_index.search(query_vector, filter_dict={"course":"llm-zoomcamp"}, num_results=5)

In [4]:
# RAG with vector search 
from rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [5]:
import os 
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("CEREBRAS_API_KEY"),
    base_url="https://api.cerebras.ai/v1"
)

In [7]:
vector_assistant = RAGVector(
    embedder = model, 
    index=vs_index,
    llm_client=openai_client,
)

In [9]:
vector_assistant.rag("the program has already began, can i still sign up")

('Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 451)